# AIE S3 — Bike Sharing Demand: Modelling & Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s3-bike-demand.ipynb)

**Regression**, ranked on **−MAE**. The same data and the same
challenge as Session 2, where a linear model scored **−138.88**.

Challenge: <https://ml-arena.com/viewchallenge/183>

---

The data does not change; the model does. The plan:

1. keep a **validation** set aside;
2. compare a linear model and a random forest on train **and**
   validation;
3. tune the forest with a **grid search**;
4. retrain the best model on all the data and submit.

---

## 0. Setup

In [ ]:
!pip install -q mlarena-sdk

---

## 1. Get the data

Your key is on your ML-Arena **Profile** page.

In [ ]:
import mlarena

API_KEY = "mlk_user_..."   # <- paste yours here
CHALLENGE_ID = 183

client = mlarena.connect(api_key=API_KEY)
client.download_dataset(CHALLENGE_ID, ".")

---

## 2. Read it

In [ ]:
import pandas as pd

X = pd.read_csv("X_train.csv")               # the hours you have a label for
y = pd.read_csv("y_train.csv")["prediction"]
X_submission = pd.read_csv("X_test.csv")     # the hours the leaderboard scores

print("X", X.shape, " X_submission", X_submission.shape)

---

## 3. Keep a validation set aside

The model never sees the validation rows during `fit`, so its score
on them estimates the leaderboard before you submit.

`X` is in time order and the leaderboard scores the hours that come
after it, so the validation set is the **last 20%** — no shuffle.

In [ ]:
n_val = int(0.2 * len(X))
X_train, y_train = X.iloc[:-n_val], y.iloc[:-n_val]
X_val,   y_val   = X.iloc[-n_val:], y.iloc[-n_val:]

print("train", X_train.shape, " val", X_val.shape)
print("mean rentals per hour: train %.0f, val %.0f" % (y_train.mean(), y_val.mean()))

The validation hours are busier (268 against 151 bikes an hour): the
system grew over the two years. Every model will do worse on them.

---

## 4. Turn it into numbers

Same preparation as Session 2: the median for missing numbers,
`"unknown"` for missing text, one-hot encoding for the text. The
medians come from the **training rows only**, and the validation
columns are aligned on the training ones.

In [ ]:
def encode(frame, medians, columns=None):
    f = frame.drop(columns=["id"]).fillna(medians)
    for c in f.columns:
        if not pd.api.types.is_numeric_dtype(f[c]):
            f[c] = f[c].fillna("unknown")
    out = pd.get_dummies(f)
    return out if columns is None else out.reindex(columns=columns, fill_value=0)


medians = X_train.median(numeric_only=True)
X_train_enc = encode(X_train, medians)
X_val_enc = encode(X_val, medians, X_train_enc.columns)
X_train_enc.shape, X_val_enc.shape

---

## 5. Two models, two scores

Each model is fitted on the training rows, then scored twice: on
those same rows (**train MAE**) and on the validation rows it has
never seen (**val MAE**).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error


def evaluate(model):
    model.fit(X_train_enc, y_train)
    train_mae = mean_absolute_error(y_train, model.predict(X_train_enc))
    val_mae = mean_absolute_error(y_val, model.predict(X_val_enc))
    print(f"train MAE {train_mae:6.1f} | val MAE {val_mae:6.1f} | {model}")


evaluate(LinearRegression())
evaluate(RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1))

**What happens?**

- Which model is better on validation?
- The forest is almost perfect on the training rows. Is it as good on
  the validation rows? What has it learned?
- The linear model is bad on both. What does that tell you?

**Try other hyperparameters.** Change the forest below, re-run it, and
watch **both** numbers:

- `max_depth` — how deep each tree can grow (5, 10, 20, …);
- `min_samples_leaf` — the fewest training hours in a leaf (5, 20, …);
- `max_features` — the share of columns each split looks at (0.3, 0.5, …);
- `n_estimators` — the number of trees. Does it change the gap?

In [ ]:
evaluate(RandomForestRegressor(n_estimators=500, max_depth=5, random_state=0, n_jobs=-1))

---

## 6. Grid search

`GridSearchCV` does what you just did, for every combination in a
grid: fit on train, score on validation.

- `PredefinedSplit` makes it use **our** split: `-1` marks the
  training rows, `0` the validation rows.
- `return_train_score=True` keeps the train score too.
- scikit-learn maximises a score, so the MAE comes back negated
  (`neg_mean_absolute_error`).

12 forests of 500 trees: expect a minute or two on Colab.

In [ ]:
from sklearn.model_selection import GridSearchCV, PredefinedSplit

grid = {
    "max_depth": [5, 10, 15, None],
    "min_samples_leaf": [1, 5, 20],
}
split = PredefinedSplit([-1] * len(X_train_enc) + [0] * len(X_val_enc))

search = GridSearchCV(
    RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1), grid,
    cv=split, scoring="neg_mean_absolute_error", return_train_score=True,
    refit=False,   # section 7 retrains the winner on all the data
)
search.fit(pd.concat([X_train_enc, X_val_enc]), pd.concat([y_train, y_val]))
print("best:", search.best_params_)

In [ ]:
results = pd.DataFrame(search.cv_results_)
results["train MAE"] = -results["mean_train_score"]
results["val MAE"] = -results["mean_test_score"]
cols = ["param_max_depth", "param_min_samples_leaf", "train MAE", "val MAE"]
results[cols].sort_values("val MAE").round(1)

Read it from the bottom up:

- `max_depth=5` and `min_samples_leaf=20` are bad on **both** columns:
  the trees are too simple. That is **underfitting**, like the linear
  model.
- The forest of section 5 (`None`, `1`) scores 8 on train and 53 on
  validation: it has memorised the training hours. That is
  **overfitting**.
- The winner, `max_depth=15`, is worse on train (11 against 8) and
  better on validation (52.6 against 53.0). **Choose on the validation
  column**, never on the train one.
- The first two rows are within 0.5 MAE of each other. A gap that
  small is noise: on the leaderboard they score about the same.

---

## 7. Retrain on all the data and submit

The grid chose the hyperparameters. Retrain them on **all** 13,903
labelled hours: the validation hours are the most recent ones, the
closest to the hours the leaderboard scores.

In [ ]:
medians_all = X.median(numeric_only=True)
X_all_enc = encode(X, medians_all)
X_submission_enc = encode(X_submission, medians_all, X_all_enc.columns)

best_model = RandomForestRegressor(n_estimators=500, random_state=0, n_jobs=-1,
                                   **search.best_params_)
best_model.fit(X_all_enc, y)

submission = pd.DataFrame({"id": X_submission["id"],
                           "prediction": best_model.predict(X_submission_enc)})
submission.to_csv("submission.csv", index=False)
submission.head()

In [ ]:
result = client.submit(challenge_id=CHALLENGE_ID, files=["submission.csv"])
print(result)

In [ ]:
client.leaderboard(CHALLENGE_ID).head(10)

About **−48**, against **−138.88** for the Session 2 line: the same
columns, another model.

---

## 8. Your turn: other models

Same protocol for other model families: fit on train, compare train
and validation MAE, then a grid search on the promising ones. No code
this time.

- `Ridge` and `Lasso` — a linear model with a penalty, `alpha`;
- `KNeighborsRegressor` — `n_neighbors`;
- `SVR` — `C`, `gamma`, `kernel` (slow on 11,000 rows: start on a
  subsample);
- `MLPRegressor` — `hidden_layer_sizes`, `alpha`, `max_iter`;
- `HistGradientBoostingRegressor` — `learning_rate`, `max_iter`,
  `max_leaf_nodes`.

All of them except the boosting need a `StandardScaler` in front:
`make_pipeline(StandardScaler(), SVR())`. For each one: does it
underfit or overfit? Does the grid search help?

Improve your **validation MAE**. When a model clearly beats the
forest, retrain it on all the data and submit — the leaderboard is a
check, not a search space.